In [3]:
import ast
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

datasets = Path("/nas/cee-water/cjgleason/data")
era5_dir = datasets / "ERA5-Land/sub_basin_timeseries"
swot_lake_dir = datasets / 'hydrocron' / 'lake'

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

matchups = gpd.read_file(metadata_dir / "All_MERIT_matchups.gpkg").set_index('comid')
matchups.index = matchups.index.astype(str)
 
# Safely convert stringified lists/dicts back to Python objects
def safe_literal_eval(df, col):
    df[col] = df[col].apply(lambda x: ast.literal_eval(x))
    return df
    
for col in ["mb_values", "lake_reach_ids", "lake_pld_ids"]:
    matchups = safe_literal_eval(matchups, col)

In [5]:
from data import BasinDeltaTable

root_dir = save_dir / 'deltalakes' / 'training'
store = BasinDeltaTable(root_dir)

processed_basins = store.get_processing_status(source='swot_river')
to_process = matchups[~matchups.index.isin(processed_basins['subbasin'])]
to_process

,outlet,outlet_id,total_area,unitarea,reservoir,custom,reach_id,sword_area,sword_distance,lake_reach_ids,...,longitude,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider,hybas_area_diff,geometry
comid,,,,,,,,,,,,,,,,,,,,,
71018502,POINT (-109.765 53.598333333333336),ECCC-05KD003,448.2,448.3,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.064249,"MULTIPOLYGON (((-109.80208 53.67208, -109.8004..."
71018507,POINT (-113.54666666666667 53.455000000000005),ECCC-05KD003,380.8,380.8,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.015892,"MULTIPOLYGON (((-113.50458 53.16125, -113.5054..."
71018653,POINT (-108.75333333333333 53.46916666666667),ECCC-05KD003,972.1,971.9,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.035666,"MULTIPOLYGON (((-108.22458 53.36792, -108.2254..."
71018658,POINT (-93.52833333333334 53.63),ECCC-04AB001,682.9,682.9,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.026015,"MULTIPOLYGON (((-93.66958 53.62875, -93.66458 ..."
71018663,POINT (-94.16666666666667 53.545833333333334),ECCC-04AB001,348.0,348.0,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.004587,"MULTIPOLYGON (((-93.96708 53.47542, -93.96708 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-410401112134801,POINT (-112.1875 41.0625),USGS-410401112134801,9900.3,296.5,False,True,NaN,NaN,NaN,[],...,-112.230256,2003-10-01,2025-09-08,0.006230,45.873291,11.263905,7165.0,usgs,-0.002093,"MULTIPOLYGON (((-112.21458 40.96042, -112.2154..."
USGS-50035000,POINT (-66.4592 18.3217),USGS-50037000,345.7,345.7,False,True,NaN,NaN,NaN,[],...,-66.459568,1950-01-01,2025-09-08,0.240693,1857.585136,7.406389,25691.0,usgs,-0.091982,"MULTIPOLYGON (((-66.49208 18.29625, -66.49208 ..."
USGS-50037000,POINT (-66.5 18.3983),USGS-50037000,429.1,83.4,False,True,NaN,NaN,NaN,[],...,-66.496560,2019-06-13,2025-09-08,1.265763,911.802460,11.017469,2247.0,usgs,0.167805,"MULTIPOLYGON (((-66.50125 18.33125, -66.50125 ..."


In [6]:
import pyarrow.dataset as ds
import itertools 

lake_ids = list(itertools.chain.from_iterable(matchups['lake_reach_ids']))
river_reaches = matchups['reach_id'].dropna().astype(int).to_list()
all_reaches = lake_ids + river_reaches

fields=[
    'reach_id', 'time','wse', 'wse_u', 'wse_r_u',
    'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u',
    'width', 'width_u',
    'area_total', 'area_tot_u', 'area_detct', 'area_det_u', 'area_wse',
    'layovr_val', 'node_dist', 'loc_offset', 'xtrk_dist',
    'reach_q', 'reach_q_b', 'dark_frac', 'ice_clim_f', 'partial_f',
    'obs_frac_n', 'xovr_cal_q', 'dry_trop_c', 'wet_trop_c', 'iono_c', 'xovr_cal_c'
]

reach_dir  = datasets / 'hydrocron' / 'reach'
list(reach_dir.glob('*.parquet'))

swot = []
for reach_file in reach_dir.glob('*_hydrocron_reach.parquet'):
    dataset = ds.dataset(reach_file, format="parquet")
    table = dataset.to_table(
        columns=fields,
        filter=(ds.field("reach_id").isin(all_reaches))
    )
    swot.append(table.to_pandas())
    
all_swot = pd.concat(swot)
all_swot = all_swot[all_swot['wse'] != -999999999999.0]
all_swot['d_wse'] = all_swot['wse'] - all_swot.groupby('reach_id')['wse'].transform('median')
all_swot['d_width'] = all_swot['width'] - all_swot.groupby('reach_id')['width'].transform('median')

all_swot = all_swot.set_index(['reach_id','time'])

In [13]:
processed_basins = store.get_processing_status(source='swot_river')
to_process = matchups[~matchups.index.isin(processed_basins['subbasin'])]

def get_swot_r(reach_id):
    if np.isnan(reach_id):
        return None
        
    if reach_id in all_swot.index.get_level_values('reach_id'):
        swot = all_swot.xs(reach_id, level='reach_id')
        swot.index = pd.to_datetime(swot.index).tz_localize('UTC')
        return swot.add_suffix('_river')


for comid, row in tqdm(to_process.iterrows(), total=len(to_process)):
    reach_id = row['reach_id']

    swot_r_df = get_swot_r(reach_id)
    store.upsert_dynamic(
        basin = row['outlet_id'], 
        subbasin = comid,
        source = 'swot_river', 
        data = swot_r_df
    )

  4%|▍         | 1141/27253 [16:47<6:24:09,  1.13it/s]


KeyboardInterrupt: 

In [15]:
store.compact()

Compacting dynamic_data...
Files added: 0, Files removed: 0.
Compacting processing metadata...
Files added: 0, Files removed: 0.


In [ ]:
glow_dir = datasets / "GLOW-S" / "daily_reach_aggregated"

glow_dfs = []
for region_file in glow_dir.glob('*_daily_median.parquet'):
    glow_dfs.append(pd.read_parquet(region_file))
                    
glow = pd.concat(glow_dfs)
glow

In [ ]:
matchups['reach_id']

In [ ]:
reach_id

In [ ]:
def get_glow_s(merit_reaches):
    glow_mask = glow.index.get_level_values('COMID').isin(merit_reaches)
    if glow_mask.any():
        glow_ix = glow[glow_mask].groupby('date').median()
        glow_ix.index = pd.to_datetime(glow_ix.index).tz_localize('UTC')
        return glow_ix.rename(columns={'width':'glow_width'})
    return pd.DataFrame(columns = ['glow_width'])

to_process = matchups.index.unique()
for comid, row in tqdm(matchups.iterrows(), total=len(matchups)):
    merit_basins = row['mb_values']
    if len(merit_basins) == 0:
        continue
        
    glow_df = get_glow_s(merit_basins)
    if not glow_df.empty:
        store.upsert_dynamic(
            basin = row['terminal_comid'], 
            subbasin = comid,
            source = 'glow_s', 
            data = glow_df
        )

In [ ]:
# pld_fields = [
#     "n_overlap", "wse", "wse_u", "wse_r_u", "wse_std",
#     "area_total", "area_tot_u", "area_detct", "area_det_u",
#     "dark_frac", "xovr_cal_q",  "layovr_val", "xtrk_dist",
#     "quality_f",  "partial_f", "ice_clim_f",
#     "dry_trop_c", "wet_trop_c", "iono_c", "xovr_cal_c"
# ]






# def get_swot_l(pld_ids: list):
#     lake_dfs = []
#     for pld_id in pld_ids:
#         path = Path(swot_lake_dir / f"{pld_id}.parquet")
#         if path.is_file():
#             swot = pd.read_parquet(path)[pld_fields]
#             swot = swot.replace(-999999999999.0, np.nan)
#             swot.dropna(subset=['wse', 'area_total'])
    
#             wse = swot['wse']
#             area = swot['area_total']
#             swot['d_wse'] = wse - wse.median()
#             swot['d_area'] = area - area.median()
#             swot['d_volume'] = swot['d_wse'] * (0.5*(area + area.median()))
#             lake_dfs.append(swot)
#         else:
#             lake_dfs.append([])

#     df_lens = [len(d) for d in lake_dfs]
#     if any(l>0 for l in df_lens):
#         swot = lake_dfs[np.argmax(df_lens)]
#         swot.index = swot.index.normalize().tz_convert('UTC')
#         return swot
#     else:
#         new_fields = ['d_wse', 'd_area', 'd_volume']
#         return pd.DataFrame(columns = pld_fields + new_fields)
    

# def get_gauge(site_id):
#     gauge = facade.get_daily_values(site_id, t1, t2).droplevel('site_id')
#     gauge.index = gauge.index.tz_localize('UTC')
#     return gauge[['discharge']]


In [ ]:
# %%
date_range = pd.date_range(start=t1, end=t2, freq='D')
# to_process = get_reaches_to_process()
to_process = matchups.index.unique()

for hybas_id in tqdm(to_process, total=len(to_process), desc="Writing files"):
    matchups_ix = matchups.loc[hybas_id]
    
    swot_r_df = get_swot_r(matchups_ix['reach_id']).add_suffix('_river')
    zds.add_dynamic_source(hybas_id, 'swot_river', swot_r_df)

    # swot_l_df = get_swot_l(matchups_ix['lake_pld_ids']).add_suffix('_lake')
    # zds.add_dynamic_source(hybas_id, 'swot_lake', swot_l_df)
    
    glow_df = get_glow_s(matchups_ix['mb_values'])
    zds.add_dynamic_source(hybas_id, 'glow_s', glow_df)

    # if matchups_ix['custom'] and not hybas_id=='outlet':
    #     gauge_df = get_gauge(hybas_id)
    # else:
    #     gauge_df = pd.DataFrame(columns = ['discharge'])
    # gauge_df['sp_discharge'] = gauge_df['discharge'] / matchups_ix['total_area']
    # zds.add_dynamic_source(hybas_id, 'gauge', gauge_df)

In [ ]:
zds.add_dynamic_source('swot_river', swot_r_df, hybas_id)

In [ ]:
gauge_df2 = pd.DataFrame(columns = ['discharge'])
gauge_df2['sp_discharge'] = gauge_df2['discharge'] / matchups_ix['total_area']

gauge_df2

In [ ]:
(gauge_df / matchups_ix['total_area']).plot()

In [ ]:
matchups_ix['total_area']

In [ ]:
hybas_id

In [ ]:
to_process